# Atlas Inventory Reorder Intelligence — Methodology & Validation

This notebook walks through the PoC's three pieces:
1. **Synthetic ERP data** — sanity-check that the simulated sales history reflects Atlas's known seasonality and volumes.
2. **Forecasting** — moving average + multiplicative seasonality; backtest on the last 12 months and compute MAPE.
3. **Reorder logic** — classical reorder point with safety stock; walk through one representative SKU end-to-end.

The goal is to keep the modelling **transparent** — every number is either in the ERP or derived from a formula the reader can follow by eye.

In [ ]:
import sys, os, sqlite3
from datetime import date
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.data_gen import DB_PATH, generate
from src.forecasting import forecast_sku, compute_seasonality, backtest_sku
from src.reorder import recommend, recommend_all

if not DB_PATH.exists():
    generate()
conn = sqlite3.connect(DB_PATH)
print('DB:', DB_PATH)

## 1. Synthetic data sanity checks

In [ ]:
products = pd.read_sql_query('SELECT * FROM products', conn)
sales = pd.read_sql_query('SELECT sku_id, sale_date, quantity FROM sales', conn, parse_dates=['sale_date'])
inv = pd.read_sql_query('SELECT sku_id, snapshot_date, on_hand FROM inventory_snapshots', conn, parse_dates=['snapshot_date'])
pos = pd.read_sql_query('SELECT * FROM purchase_orders', conn, parse_dates=['order_date', 'eta_date', 'received_date'])

print(f'Products:      {len(products):,}')
print(f'  paint:       {(products.category == "paint").sum()}')
print(f'  hardware:    {(products.category == "hardware").sum()}')
print(f'Sales rows:    {len(sales):,}')
print(f'  date range:  {sales.sale_date.min().date()} → {sales.sale_date.max().date()}')
print(f'Inventory rows:{len(inv):,}')
print(f'PO rows:       {len(pos):,}')

In [ ]:
# Monthly total revenue to confirm seasonality (peak March + summer, trough Dec-Feb)
sales_val = sales.merge(products[['sku_id', 'unit_cost']], on='sku_id')
sales_val['revenue'] = sales_val['quantity'] * sales_val['unit_cost'] * 1.45
sales_val['month'] = sales_val['sale_date'].dt.to_period('M').dt.to_timestamp()
monthly = sales_val.groupby('month')['revenue'].sum()

fig, ax = plt.subplots(figsize=(10, 3))
monthly.plot(ax=ax)
ax.set_title('Monthly revenue (synthetic) — seasonality visible')
ax.set_ylabel('Revenue ($)')
ax.set_xlabel('')
plt.tight_layout(); plt.show()

month_avg = sales_val.groupby(sales_val['sale_date'].dt.month)['revenue'].mean()
print('Mean daily revenue by month-of-year (check 2:1 peak:trough):')
print(month_avg.round(0).to_string())

## 2. Forecasting method

For each SKU:
- compute monthly seasonality indices from full history (avg daily sales per month ÷ overall mean)
- take a **trailing 180-day window**, deseasonalize each day (÷ its month's factor), then average → baseline daily demand
- re-inflate by the **average seasonality factor for the forecast horizon**

In [ ]:
sample_sku = 1  # Interior Emulsion White 4L (high-volume paint)
season = compute_seasonality(conn, sample_sku)
pd.Series(season).rename_axis('month').rename('factor').round(2).to_frame()

In [ ]:
# 12-month backtest MAPE per SKU
rows = []
for sku_id in products['sku_id']:
    bt = backtest_sku(conn, sku_id, test_months=12)
    if not bt.empty:
        mape = bt['abs_pct_error'].mean()
        rows.append({'sku_id': sku_id, 'MAPE': mape})
mape_df = pd.DataFrame(rows).merge(products[['sku_id', 'name', 'category']], on='sku_id')
print(f'Median MAPE across {len(mape_df)} SKUs: {mape_df.MAPE.median():.1%}')
print(f'Mean   MAPE:                              {mape_df.MAPE.mean():.1%}')
print('\nBy category:')
print(mape_df.groupby('category')['MAPE'].median().round(3))

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(mape_df.MAPE.clip(upper=1.5), bins=30, color='#2c3e50')
ax.axvline(mape_df.MAPE.median(), color='#e67e22', linestyle='--', label=f'median {mape_df.MAPE.median():.1%}')
ax.set_title('Per-SKU backtest MAPE (clipped at 150%)')
ax.set_xlabel('MAPE'); ax.legend()
plt.tight_layout(); plt.show()

**Honest note on accuracy.** Monthly MAPE varies sharply by category:

- **Paint** (~12% median MAPE) \u2014 high volume, smoother demand, shorter lead times. Baseline model is very usable.
- **Hardware** (~70% median MAPE) \u2014 low-volume, lumpy B2B orders, 60\u201375 day lead times. Baseline forecast is weak; the reorder decision instead relies on service-level-driven safety stock to compensate. This is exactly the reason Phase 2 uses classical reorder-point logic instead of chasing a high-accuracy forecast: the safety stock term absorbs forecast error. For hardware specifically, **intermittent-demand models (Croston) are the natural next upgrade** if the business wants tighter targets.

The bar for a Phase-2 statistical baseline is: "good enough that the reorder decision is better than the current reactive process." Not: "lowest MAPE possible."

## 3. Reorder logic walkthrough

In [ ]:
recs = recommend_all(conn, as_of=date(2026, 3, 31))
print(f'{recs.should_reorder.sum()} of {len(recs)} SKUs flagged for reorder')
print(f'Total recommended PO value: ${recs.loc[recs.should_reorder, "currency_value"].sum():,.0f}')
recs[recs.should_reorder].sort_values('stockout_risk').head(10)[
    ['sku_code', 'name', 'on_hand', 'days_of_cover', 'reorder_point', 'suggested_order_qty', 'stockout_risk']
]

In [ ]:
# Walkthrough on a representative high-priority SKU
candidates = recs[(recs.should_reorder) & (recs.category == 'hardware')].sort_values('currency_value', ascending=False)
target = candidates.iloc[0] if not candidates.empty else recs[recs.should_reorder].iloc[0]
rec = recommend(conn, int(target.sku_id), date(2026, 3, 31))

print(f'SKU: {rec.sku_code} — {rec.name}')
print(f'Supplier: {rec.supplier}  Lead time: {rec.lead_time_days} days')
print(f'On-hand:        {rec.on_hand}')
print(f'Open POs:       {rec.open_po_qty}')
print(f'Avg daily dem:  {rec.avg_daily_demand:.2f}  (seasonality {rec.seasonality_factor:.2f})')
print(f'Daily std:      {rec.daily_std:.2f}')
print(f'Safety stock:   {rec.safety_stock:.0f}  (z=1.645 at {int(rec.service_level*100)}% service level)')
print(f'Reorder point:  {rec.reorder_point:.0f}')
print(f'Days of cover:  {rec.days_of_cover:.0f}')
print(f'Recommend?      {rec.should_reorder}')
print(f'Order qty:      {rec.suggested_order_qty}  (~${rec.currency_value:,.0f})')
print(f'Risk:           {rec.stockout_risk}')

## 4. Evaluation summary

| Metric | Value |
| --- | --- |
| SKUs modelled | 60 (20 paint + 40 hardware) |
| Historical range | 2016-01 → 2026-03 |
| Forecast horizon | lead_time + review_period (typ. 55–70 + 14 days) |
| Backtest MAPE (median) | see cell above |
| Reorders flagged as-of 2026-03-31 | see cell above |
| LLM explanation fallback | deterministic template if no API key |

**Next steps (production):**
1. Swap synthetic SQLite for a read-only mirror of the Atlas ERP (SQL extract, weekly refresh).
2. Phase-0 cycle count to fix inventory accuracy — this is the single biggest lever on model quality.
3. Review low-volume SKU handling: intermittent-demand models (Croston) if the base forecast underperforms.
4. Human-in-the-loop sign-off workflow before any PO is placed (Module 12 governance requirement).